# Evaluation 00 — Real-label and frozen-split preflight

Run this **before** GPU evaluation. Review all suspect Skin Cancer and Normal images first. This notebook only checks reviewed metadata; it **does not diagnose** disease. No test results are generated.

**Saved evidence:** `notebook_console.log` captures execution/tracebacks, `run_log.jsonl` records start and completed artifact SHA-256 checksums, and each analysis writes its reported CSV/JSON or graphic files into the configured Kaggle output directory. Keep per-image records and raw console text private; export only reviewed aggregate reports to GitHub.


In [ ]:
# Common Kaggle bootstrap: either attach DermSynReal source as Kaggle Input,
# or set CLONE_REPO_IF_MISSING=True and enable Kaggle Internet.
from pathlib import Path
import subprocess,sys,os,json
REPO_ROOT=Path("/kaggle/input/EDIT_REPO_DATASET/DermSynReal")
CLONE_REPO_IF_MISSING=True
EXPECTED_GIT_SHA=""  # Recommended: set a full pinned SHA for reproducible final runs.
if not (REPO_ROOT/"src/dermsynreal/evaluation.py").is_file():
    if not CLONE_REPO_IF_MISSING:
        raise FileNotFoundError("Attach the repository and change REPO_ROOT, or enable Git clone.")
    REPO_ROOT=Path("/kaggle/working/DermSynReal")
    if not (REPO_ROOT/".git").is_dir():
        subprocess.check_call(["git","clone","https://github.com/shanmukhdatta/DermSynReal.git",str(REPO_ROOT)])
if EXPECTED_GIT_SHA:
    if not (REPO_ROOT/".git").is_dir():
        raise ValueError("For pinned SHA verification attach a Git checkout or enable clone")
    actual=subprocess.check_output(["git","-C",str(REPO_ROOT),"rev-parse","HEAD"],text=True).strip()
    if actual!=EXPECTED_GIT_SHA:
        raise ValueError(f"Repository SHA {actual} differs from pinned SHA {EXPECTED_GIT_SHA}")
elif (REPO_ROOT/".git").is_dir():
    print("Current Git SHA:",subprocess.check_output(["git","-C",str(REPO_ROOT),"rev-parse","HEAD"],text=True).strip())
    print("WARNING: pin EXPECTED_GIT_SHA for final research outputs")
sys.path.insert(0,str(REPO_ROOT/"src"))
print("DermSynReal source:",REPO_ROOT)


## Configure approved source and frozen split CSV paths
Expected review CSV columns: `image_id,source,class_name,original_diagnosis,reviewer_decision,reviewer_id,image_path`.
Expected frozen split CSV: `image_id,image_path,label,split,group_id`.

In [ ]:
REVIEWED_MANIFEST="/kaggle/input/EDIT_REAL_LABELS/reviewed_real_manifest.csv"
FROZEN_SPLIT_CSV="/kaggle/input/EDIT_REAL_SPLITS/real_split.csv"
OUT_DIR="/kaggle/working/dermsynreal_eval_00"
RUN_AUDIT=False  # Set True only after actual image-based label review.

In [ ]:
# Preserve a complete Kaggle notebook console and structured start event.
# This log can contain image file paths or case notes; keep it out of public GitHub.
from dermsynreal.kaggle_runs import start_notebook_logging,append_run_log
EVALUATION_OUTPUT=Path(OUT_DIR)
PRIVATE_EVALUATION_LOG=start_notebook_logging(EVALUATION_OUTPUT,
    kind='evaluation_00',config={
        'notebook':'00_Real_Label_And_Split_Preflight.ipynb','data_verified':False,
        'run_enabled':bool(RUN_AUDIT)},repo_root=REPO_ROOT)
print('Saved notebook console:',PRIVATE_EVALUATION_LOG)


In [ ]:
from dermsynreal.label_gate import write_audit
from dermsynreal.evaluation import CLASSES,_label
import csv,hashlib
out=Path(OUT_DIR)
if not RUN_AUDIT:
    print("Not executed: complete the clinical label review, correct input paths, then set RUN_AUDIT=True.")
else:
    audit=write_audit(REVIEWED_MANIFEST,out/"labels")
    with open(FROZEN_SPLIT_CSV,newline="",encoding="utf-8-sig") as f:
        reader=csv.DictReader(f)
        needed={"image_id","image_path","label","split","group_id"}
        assert needed<=set(reader.fieldnames or []),f"Missing split fields: {needed-set(reader.fieldnames or [])}"
        rows=list(reader)
    ids=set(); groups={};by_split={x:[] for x in ("train","val","test")}
    with open(out/"labels/approved_for_split.csv",newline="",encoding="utf-8") as f:
        approved={r["image_id"]:r for r in csv.DictReader(f)}
    for r in rows:
        i=r["image_id"]
        assert i and i not in ids,f"Duplicate/missing ID: {i}"
        ids.add(i)
        assert i in approved,f"Split has unapproved ID: {i}"
        assert _label(r["label"])==_label(approved[i]["class_name"])
        assert r["group_id"],f"Missing protected group for {i}"
        assert r["split"] in by_split
        assert r["group_id"] not in groups or groups[r["group_id"]]==r["split"],"Cross-split duplicate group"
        groups[r["group_id"]]=r["split"]
        by_split[r["split"]].append(r)
    for name,rows_in in by_split.items():
        counts={c:sum(_label(r["label"])==c for r in rows_in) for c in CLASSES}
        print(name,len(rows_in),"verified groups",len({r["group_id"] for r in rows_in}),counts)
        assert all(counts.values()),f"Missing target class in {name}"
    proof={"passed":True,"real_split_sha":hashlib.sha256(Path(FROZEN_SPLIT_CSV).read_bytes()).hexdigest(),
           "split_n":{name:len(v) for name,v in by_split.items()},"group_n":len(groups),
           "note":"This is structural validation, not clinical verification or complete leakage detection."}
    out.mkdir(parents=True,exist_ok=True)
    (out/"frozen_split_preflight.json").write_text(json.dumps(proof,indent=2)+"\n")
    print("PASS metadata gate; inspect duplicates and reviewer evidence independently.",proof)


**Next:** use the *same* frozen real-test manifest for every E0–E5 comparison, and preserve the split SHA. Do not edit labels after seeing test predictions.

In [ ]:
# Record checksums only for files actually created by this executed run.
if RUN_AUDIT:
    _files=[Path(OUT_DIR)/'labels/label_audit_summary.json',Path(OUT_DIR)/'frozen_split_preflight.json']
    _log=append_run_log(EVALUATION_OUTPUT,kind='evaluation_00_finished',
        config={'notebook':'00_Real_Label_And_Split_Preflight.ipynb','status':'completed_artifacts_written'},
        artifacts=[p for p in _files if p.is_file()],repo_root=REPO_ROOT)
    print('Evidence hashes:',_log['artifacts_sha256'])
